# Mann-Whitney U and DeLong Analyses

This notebook runs the standalone analysis scripts in `scripts/mwu_delongs_scripts/`. The statistical implementation remains in those Python files rather than being duplicated here.

Expected local directories beside this notebook:

- `scRNASeq/`: input `.h5ad` datasets
- `scores/<dataset>/`: activity-score parquet files
- `common_tfs/`: common-TF tables
- `results/`: generated MWU and DeLong TSV files

Run the setup cell first. The three analysis sections can then be run independently or sequentially. Missing dataset or score files are handled by the scripts' existing skip logic.


## Setup

Locate the local reproduction directory and define a helper that streams each script's output into the notebook.


In [ ]:
from pathlib import Path
import subprocess
import sys

def find_analysis_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "utility_functions.py").is_file():
            return candidate
        nested = candidate / "reproduce" / "Reproduce scRNASeq Results"
        if (nested / "scripts" / "utility_functions.py").is_file():
            return nested
    raise FileNotFoundError("Could not locate the Reproduce scRNASeq Results directory")

analysis_dir = find_analysis_dir(Path.cwd().resolve())
scripts_dir = analysis_dir / "scripts" / "mwu_delongs_scripts"

def run_analysis_script(filename: str) -> None:
    script_path = scripts_dir / filename
    if not script_path.is_file():
        raise FileNotFoundError(f"Missing analysis script: {script_path}")

    print(f"Running {script_path.name}")
    print(f"Working directory: {analysis_dir}")

    process = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=analysis_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [sys.executable, str(script_path)])

    print(f"Completed {script_path.name}")

print(f"Analysis directory: {analysis_dir}")
print(f"Scripts directory: {scripts_dir}")


## 1. Compare activity-inference methods

Run MWU and top-two DeLong comparisons for z-aggregate, VIPER, ULM, and z-score using the CausalPath prior and uniform weights.


In [ ]:
run_analysis_script("Methods_MWU-Delongs.py")


## 2. Compare prior networks

Run MWU and top-two DeLong comparisons for the CausalPath, CollecTRI, DoRothEA, and ensemble prior networks.


In [ ]:
run_analysis_script("Priors_MWU-Delongs.py")


## 3. Compare network-weighting strategies

Run MWU and top-two DeLong comparisons for uniform, correlation, specificity, and non-zero-rate weighting.


In [ ]:
run_analysis_script("Weights_MWU-Delongs.py")
